## TabNet: Attentive Interpretable Tabular Learning

"We propose a novel high-performance and interpretable canonical deep tabular data learning architecture, TabNet. TabNet uses sequential attention to choose which features to reason from at each decision step, enabling interpretability and more efficient learning as the learning capacity is used for the most salient features. We demonstrate that TabNet outperforms other neural network and decision tree variants on a wide range of non-performance-saturated tabular datasets and yields interpretable feature attributions plus insights into the global model behavior. Finally, for the first time to our knowledge, we demonstrate self-supervised learning for tabular data, significantly improving performance with unsupervised representation learning when unlabeled data is abundant." 

[TabNet Paper](https://arxiv.org/abs/1908.07442)


## Install PyTorch TabNet

In [1]:
!pip install pytorch-tabnet

     |████████████████████████████████| 44 kB 1.4 MB/s 
You should consider upgrading via the '/opt/conda/bin/python3.7 -m pip install --upgrade pip' command.


## Parameters

In [2]:
# To ignore warinings
import warnings
warnings.filterwarnings('ignore')

In [3]:
NUM_FOLDS = 7
SEED = 42

## TabNet Parameters

In [4]:
## TabNet Parameters
MAX_EPOCH = 300
N_D = 6 
N_A = 6 
N_STEPS = 2
GAMMA = 1.1
LAMBDA_SPARSE = 1e-4
OPT_LR = 1e-4
OPT_WEIGHT_DECAY = 1e-5
OPT_MOMENTUM = 0.9
MASK_TYPE = "entmax"
SCHEDULER_MIN_LR = 1e-6
SCHEDULER_FACTOR = 0.9
DEVICE_NAME = "cuda"

BATCH_SIZE = 32

## Imports Libs

In [5]:
import torch
from torch import nn
from pytorch_tabnet.tab_model import TabNetRegressor

from tqdm.notebook import tqdm
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import KFold

from sklearn.metrics import mean_squared_error

import numpy as np
import pandas as pd 

import os
import random
import sys
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"


def seed_everything(seed_value):
    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    os.environ['PYTHONHASHSEED'] = str(seed_value)
    
    if torch.cuda.is_available(): 
        torch.cuda.manual_seed(seed_value)
        torch.cuda.manual_seed_all(seed_value)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        
seed_everything(SEED)

## Import Data

In [6]:
import pandas as pd

train = pd.read_csv("/kaggle/input/zelestra/train.csv")

train = train.drop(columns=['id'])
test=pd.read_csv("/kaggle/input/zelestra/test.csv").drop(columns=['id'])

In [7]:
train.dtypes

temperature           float64
irradiance            float64
humidity               object
panel_age             float64
maintenance_count     float64
soiling_ratio         float64
voltage               float64
current               float64
module_temperature    float64
cloud_coverage        float64
wind_speed             object
pressure               object
string_id              object
error_code             object
installation_type      object
efficiency            float64
dtype: object

In [8]:
columns = train.columns.to_list()
features = columns[:-1]
target = columns[-1]

print(f'Features: {features}')
print(f'Target: {target}')

Features: ['temperature', 'irradiance', 'humidity', 'panel_age', 'maintenance_count', 'soiling_ratio', 'voltage', 'current', 'module_temperature', 'cloud_coverage', 'wind_speed', 'pressure', 'string_id', 'error_code', 'installation_type']
Target: efficiency


In [9]:
obj_num_cols=train.select_dtypes(include='object').columns[:-3]

In [10]:
obj_num_cols

Index(['humidity', 'wind_speed', 'pressure'], dtype='object')

In [11]:
for col in obj_num_cols:
    train[col] = pd.to_numeric(train[col], errors='coerce')

In [12]:
train.dtypes

temperature           float64
irradiance            float64
humidity              float64
panel_age             float64
maintenance_count     float64
soiling_ratio         float64
voltage               float64
current               float64
module_temperature    float64
cloud_coverage        float64
wind_speed            float64
pressure              float64
string_id              object
error_code             object
installation_type      object
efficiency            float64
dtype: object

In [13]:
for col in obj_num_cols:
    test[col] = pd.to_numeric(test[col], errors='coerce')

In [14]:
X_train = train[features]
y_train = train[target]

In [15]:
X_train.dtypes

temperature           float64
irradiance            float64
humidity              float64
panel_age             float64
maintenance_count     float64
soiling_ratio         float64
voltage               float64
current               float64
module_temperature    float64
cloud_coverage        float64
wind_speed            float64
pressure              float64
string_id              object
error_code             object
installation_type      object
dtype: object

In [16]:
from sklearn.preprocessing import PowerTransformer
pt = PowerTransformer(method='yeo-johnson')


pt.fit(X_train[list(X_train.select_dtypes(include='float64').columns)])
X_train_pt = pt.transform(X_train[list(X_train.select_dtypes(include='float64').columns)])

print(f'Shape of X_train_pt: {X_train_pt.shape}')

Shape of X_train_pt: (20000, 12)


In [17]:
X_train.dtypes

temperature           float64
irradiance            float64
humidity              float64
panel_age             float64
maintenance_count     float64
soiling_ratio         float64
voltage               float64
current               float64
module_temperature    float64
cloud_coverage        float64
wind_speed            float64
pressure              float64
string_id              object
error_code             object
installation_type      object
dtype: object

In [18]:
pt.fit(test[list(test.select_dtypes(include='float64').columns)])
test_pt = pt.transform(test[list(test.select_dtypes(include='float64').columns)])

In [19]:
test_pt=pd.DataFrame(test_pt,columns=list(test.select_dtypes(include='float64').columns))
X_train_pt = pd.DataFrame(X_train_pt, columns=list(X_train.select_dtypes(include='float64').columns))

In [20]:
X_train_pt[list(X_train.select_dtypes(include='object').columns)]=X_train[list(X_train.select_dtypes(include='object').columns)]

In [21]:
X_train_pt.dtypes

temperature           float64
irradiance            float64
humidity              float64
panel_age             float64
maintenance_count     float64
soiling_ratio         float64
voltage               float64
current               float64
module_temperature    float64
cloud_coverage        float64
wind_speed            float64
pressure              float64
string_id              object
error_code             object
installation_type      object
dtype: object

In [22]:
test_pt[list(test.select_dtypes(include='object').columns)]=test[list(test.select_dtypes(include='object').columns)]

In [23]:
X_train_pt.dtypes

temperature           float64
irradiance            float64
humidity              float64
panel_age             float64
maintenance_count     float64
soiling_ratio         float64
voltage               float64
current               float64
module_temperature    float64
cloud_coverage        float64
wind_speed            float64
pressure              float64
string_id              object
error_code             object
installation_type      object
dtype: object

In [24]:
from sklearn.preprocessing import LabelEncoder

# Sample: list of columns to encode
cols_to_encode = list(X_train_pt.select_dtypes(include='object').columns)

le = LabelEncoder()
for col in cols_to_encode:
    # Convert all values to string to avoid mixed types (also handles NaN as 'nan')
    X_train_pt[col] = X_train_pt[col].astype(str)
    X_train_pt[col] = le.fit_transform(X_train_pt[col])

In [25]:
for col in cols_to_encode:
    test_pt[col] = test_pt[col].astype(str)
    test_pt[col] = le.fit_transform(test_pt[col])

In [26]:
target = train.pop("efficiency")
target = target.values

## Label Encoding

In [27]:
columns = test.columns

## Create TabNet Params Dictionary

In [28]:
CAT_IDX = [12, 13, 14]
CAT_DIMS=[]
for col_idx in CAT_IDX:
    n_unique = X_train_pt.iloc[:, col_idx].nunique()
    CAT_DIMS.append(n_unique)

In [29]:
CAT_DIMS

[4, 4, 4]

In [30]:
CAT_EMB_DIM=[3,3,3]

In [31]:
tabnet_params = dict(n_d=N_D, n_a=N_A, n_steps=N_STEPS, gamma=GAMMA,
                     lambda_sparse=LAMBDA_SPARSE, optimizer_fn=torch.optim.Adam,
                     mask_type=MASK_TYPE,
                     cat_idxs=CAT_IDX,           # List of categorical column indices
                     cat_dims=CAT_DIMS,           # Number of unique values per categorical
                     cat_emb_dim=CAT_EMB_DIM,     # Embedding dimensions for categoricals
                     scheduler_params=dict(mode="min",
                                           patience=20,
                                           min_lr=SCHEDULER_MIN_LR,
                                           factor=SCHEDULER_FACTOR,),
                     scheduler_fn=torch.optim.lr_scheduler.ReduceLROnPlateau,
                     verbose=10,
                     device_name=DEVICE_NAME,
                     seed=SEED
                     )

## Run Kfold with TabNet Regressor

In [32]:
X_train_pt.dtypes

temperature           float64
irradiance            float64
humidity              float64
panel_age             float64
maintenance_count     float64
soiling_ratio         float64
voltage               float64
current               float64
module_temperature    float64
cloud_coverage        float64
wind_speed            float64
pressure              float64
string_id               int64
error_code              int64
installation_type       int64
dtype: object

In [33]:
target.dtype

dtype('float64')

In [34]:
import numpy as np

X_train_pt.replace([np.inf, -np.inf], np.nan, inplace=True)
test_pt.replace([np.inf, -np.inf], np.nan, inplace=True)
from sklearn.impute import SimpleImputer

numeric_cols = X_train_pt.select_dtypes(include=[np.number]).columns

# Create the imputer (you can use 'mean', 'median', or 'most_frequent')
imputer = SimpleImputer(strategy='mean')

# Fit and transform only on numeric columns
X_train_pt[numeric_cols] = imputer.fit_transform(X_train_pt[numeric_cols])
numeric_cols = test_pt.select_dtypes(include=[np.number]).columns
test_pt[numeric_cols] = imputer.fit_transform(test_pt[numeric_cols])


In [35]:
list(X_train_pt.columns)[12:]

['string_id', 'error_code', 'installation_type']

In [36]:
X_train_pt.dtypes

temperature           float64
irradiance            float64
humidity              float64
panel_age             float64
maintenance_count     float64
soiling_ratio         float64
voltage               float64
current               float64
module_temperature    float64
cloud_coverage        float64
wind_speed            float64
pressure              float64
string_id             float64
error_code            float64
installation_type     float64
dtype: object

In [37]:
X_train_pt['string_id'].unique()

array([0., 3., 2., 1.])

In [38]:
train_oof = np.zeros((len(train)))
test_preds = 0

kf = KFold(n_splits=NUM_FOLDS, shuffle=True, random_state=SEED)

for f, (train_ind, val_ind) in tqdm(enumerate(kf.split(X_train_pt, target))):

    print(f'Fold {f}')
    train_df, val_df = X_train_pt.iloc[train_ind][columns], X_train_pt.iloc[val_ind][columns]

    train_target, val_target = target[train_ind], target[val_ind]

    print(X_train_pt.shape, train_target.shape)
    print(val_df.shape, val_target.shape)

    train_target=train_target.reshape(-1,1)
    val_target=val_target.reshape(-1,1)

    train_df      = train_df.to_numpy()
    train_target      = train_target.reshape(-1, 1)

    val_df = val_df.to_numpy()
    val_target = val_target.reshape(-1, 1)

    model = TabNetRegressor(**tabnet_params)

    model.fit(X_train=train_df,
              y_train=train_target,
              eval_set=[(val_df, val_target)],
              eval_name = ["val"],
              eval_metric = ['rmse'],#["logits_ll"],
              max_epochs=MAX_EPOCH, #20
              patience=20, batch_size=BATCH_SIZE,
              num_workers=1, drop_last=False)#,

    temp_oof = model.predict(val_df)
    train_oof[val_ind] = temp_oof.reshape(-1)
    temp_test = model.predict(test_pt.to_numpy())

    test_preds += temp_test/NUM_FOLDS     

    print(mean_squared_error(temp_oof, val_target, squared=False))




|          | 0/? [00:00<?, ?it/s]

Fold 0
(20000, 15) (17142,)
(2858, 15) (2858,)
epoch 0  | loss: 0.03249 | val_rmse: 0.11201 |  0:00:16s
epoch 10 | loss: 0.01139 | val_rmse: 0.1089  |  0:02:50s
epoch 20 | loss: 0.01137 | val_rmse: 0.10821 |  0:05:24s
epoch 30 | loss: 0.01138 | val_rmse: 0.1081  |  0:07:58s
epoch 40 | loss: 0.01132 | val_rmse: 0.10911 |  0:10:31s

Early stopping occurred at epoch 45 with best_epoch = 25 and best_val_rmse = 0.1074
0.10740063545580084
Fold 1
(20000, 15) (17143,)
(2857, 15) (2857,)
epoch 0  | loss: 0.03643 | val_rmse: 0.11746 |  0:00:15s
epoch 10 | loss: 0.01172 | val_rmse: 0.10361 |  0:02:50s
epoch 20 | loss: 0.01151 | val_rmse: 0.10232 |  0:05:24s
epoch 30 | loss: 0.01128 | val_rmse: 0.10313 |  0:07:59s

Early stopping occurred at epoch 36 with best_epoch = 16 and best_val_rmse = 0.10211
0.10210914162425905
Fold 2
(20000, 15) (17143,)
(2857, 15) (2857,)
epoch 0  | loss: 0.03179 | val_rmse: 0.11343 |  0:00:15s
epoch 10 | loss: 0.01188 | val_rmse: 0.10295 |  0:02:49s
epoch 20 | loss: 0.01

In [39]:
test_preds

array([[0.3789449 ],
       [0.5413547 ],
       [0.5225351 ],
       ...,
       [0.6148004 ],
       [0.43719363],
       [0.5479768 ]], dtype=float32)

In [40]:
temp_test = model.predict(test_pt.to_numpy())

In [41]:
temp_test

array([[0.39368427],
       [0.54829574],
       [0.51968855],
       ...,
       [0.62718177],
       [0.4181956 ],
       [0.5469151 ]], dtype=float32)

## Submit your output csv

In [42]:
submission = pd.read_csv('/kaggle/input/zelestra/test.csv')

In [43]:
temp_test.squeeze(1)

array([0.39368427, 0.54829574, 0.51968855, ..., 0.62718177, 0.4181956 ,
       0.5469151 ], dtype=float32)

In [44]:
submission_df =pd.DataFrame({'id': submission['id'].values, 'efficiency': temp_test.squeeze(1)})

In [45]:
submission_df.to_csv("etemp.csv",index=False)

In [46]:
submission_df =pd.DataFrame({'id': submission['id'].values, 'efficiency': test_preds.squeeze(1)})

In [47]:
submission_df.to_csv("avg.csv",index=False)

## Credits

* [TabNet Paper](https://arxiv.org/pdf/1908.07442.pdf)
* [TabNet PyTorch GitHub](https://github.com/dreamquark-ai/tabnet)
* [Kaggle Notebook TabNet Regressor](https://www.kaggle.com/optimo/tabnetregressor-2-0-train-infer?scriptVersionId=44853427)
* [Tunguz CV Notebook](https://www.kaggle.com/tunguz/tps-02-21-feature-importance-with-xgboost-and-shap)

## Some ideas:

* Fork the notebook and try to change some parameters and play with the model...
* Try to add some preprocessing...
* Try other encoding than label encoding...
* Ensemble your model with others to make it more diversity...

## If it was useful for you please comment! Your feedback is really important! 